In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nqa112/vizwiz-2023-edition")

print("Path to dataset files:", path)

100%|██████████| 17.5G/17.5G [04:01<00:00, 77.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1


In [ ]:
# !ls /root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json

# BLUE-1 Score

In [ ]:
import json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Đọc dữ liệu
with open('/content/vqa_test_predictions.json', 'r') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json', 'r') as f:
    ground_truths = json.load(f)

# Tạo dictionary cho ground truth để dễ tra cứu
gt_dict = {item['image']: item['answers'] for item in ground_truths}

# Khởi tạo
scores = []
smoothie = SmoothingFunction().method4

for pred in predictions:
    image_id = pred['image']
    # pred_answer = pred['answer']
    pred_answer = pred['predicted_answer']

    # Tokenize
    pred_tokens = pred_answer.lower().split()

    # Lấy các ground truth answers
    references = [ans['answer'].lower().split() for ans in gt_dict.get(image_id, []) if 'answer' in ans]

    # Tính BLEU cho từng câu
    if references:
        bleu = sentence_bleu(references, pred_tokens, smoothing_function=smoothie)
        scores.append(bleu)

# Tính trung bình BLEU score
average_bleu = sum(scores) / len(scores) if scores else 0.0
print(f"Average BLEU score: {average_bleu:.4f}")

Average BLEU score: 0.6443


# VQA Accuracy

In [ ]:
import json
from collections import Counter

# Đọc dữ liệu
with open('/content/vqa_test_predictions.json') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Tạo dict tra cứu ground truth theo image_id
gt_dict = {item['image']: item['answers'] for item in ground_truths}

accuracies = []

for pred in predictions:
    image_id = pred['image']
    # pred_answer = pred['answer'].strip().lower()
    pred_answer = pred['predicted_answer'].strip().lower()

    # Lấy các câu trả lời ground truth (chuyển về chữ thường và strip)
    gt_answers = [
        ans['answer'].strip().lower()
        for ans in gt_dict.get(image_id, [])
        if 'answer' in ans
    ]

    # Đếm số lần dự đoán trùng với các câu trả lời
    matching_count = sum(1 for ans in gt_answers if ans == pred_answer)

    # Tính accuracy chuẩn VizWiz
    acc = min(matching_count / 3, 1.0)
    accuracies.append(acc)

# Tính trung bình
average_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0.0
print(f"VizWiz-style VQA Accuracy: {average_accuracy:.4f}")

VizWiz-style VQA Accuracy: 0.5417


# METEOR

In [ ]:
# --- METEOR SCORE ---
import json
from nltk.translate.meteor_score import meteor_score
import nltk

# Tải WordNet cho METEOR
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
import json
from nltk.translate.meteor_score import meteor_score
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

# Load predictions
with open('/content/vqa_test_predictions.json') as f:
    predictions = json.load(f)

# Load ground truths
with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Map image -> answers
gt_dict = {x["image"]: x["answers"] for x in ground_truths}

meteor_scores = []

for pred in predictions:
    image_id = pred["image"]
    pred_answer = pred["predicted_answer"].lower().strip()

    # Ground truth answers
    references = [
        ans["answer"].lower().strip()
        for ans in gt_dict.get(image_id, [])
        if "answer" in ans
    ]

    if len(references) > 0:
        # Token hóa để phù hợp METEOR
        pred_tokens = pred_answer.split()
        ref_tokens_list = [ref.split() for ref in references]

        score = meteor_score(ref_tokens_list, pred_tokens)
        meteor_scores.append(score)

avg_meteor = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0
print(f"METEOR Score: {avg_meteor:.4f}")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR Score: 0.3211


# ROUGE-L

In [ ]:
# --- ROUGE-L SCORE ---
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d9d4d0b768f5a8dcdc38f0466110cf8133b8fe487a0e54afc4bb723125402635
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import json
from rouge_score import rouge_scorer

# Load predictions
with open('/content/vqa_test_predictions.json') as f:
    predictions = json.load(f)

# Load ground truths
with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

gt_dict = {x["image"]: x["answers"] for x in ground_truths}

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = []

for pred in predictions:
    image_id = pred["image"]
    pred_answer = pred["predicted_answer"].lower().strip()

    references = [
        ans["answer"].lower().strip()
        for ans in gt_dict.get(image_id, [])
        if "answer" in ans
    ]

    if len(references) > 0:
        # Lấy max ROUGE-L trong số các ground truth
        best_score = max(
            scorer.score(ref, pred_answer)["rougeL"].fmeasure for ref in references
        )
        rouge_scores.append(best_score)

avg_rougeL = sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0.0
print(f"ROUGE-L Score: {avg_rougeL:.4f}")

ROUGE-L Score: 0.6428


# CIDEr (MS-COCO)

In [ ]:
# --- CIDEr SCORE ---
!pip install git+https://github.com/salaniz/pycocoevalcap

  Cloning https://github.com/salaniz/pycocoevalcap to /tmp/pip-req-build-egwm3tra
  Running command git clone --filter=blob:none --quiet https://github.com/salaniz/pycocoevalcap /tmp/pip-req-build-egwm3tra
  Resolved https://github.com/salaniz/pycocoevalcap to commit a24f74c408c918f1f4ec34e9514bc8a76ce41ffd
  Preparing metadata (setup.py) ... done
  Created wheel for pycocoevalcap: filename=pycocoevalcap-1.2-py3-none-any.whl size=104312245 sha256=d00eef80868ee8e843c2d2a4f3a23661f0e1589f32f164fa8ac59da6647a64cd
  Stored in directory: /tmp/pip-ephem-wheel-cache-tgt_48hm/wheels/03/ce/0b/3d3fdeecb09b4f4ebcfb3ff28d27a9f5b3c1a7b73897ad122d
Successfully built pycocoevalcap


In [ ]:
import json
from pycocoevalcap.cider.cider import Cider

# Load predictions
with open('/content/vqa_test_predictions.json') as f:
    predictions = json.load(f)

# Load ground truths
with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Chuẩn hóa về format của COCO Caption Evaluator (list of strings)
gts = {}   # ground truth dictionary
res = {}   # prediction dictionary

for pred in predictions:
    image_id = pred["image"]
    answer = pred["predicted_answer"].lower().strip()
    res[image_id] = [answer]  # chỉ là list string

for item in ground_truths:
    img = item["image"]
    answers = [
        ans["answer"].lower().strip()
        for ans in item["answers"]
        if "answer" in ans
    ]
    gts[img] = answers  # list string

# Tính CIDEr
cider = Cider()
score, scores = cider.compute_score(gts, res)

print(f"CIDEr Score: {score:.4f}")


CIDEr Score: 0.8085


# Catergories

In [ ]:
import json
from collections import defaultdict

# Load predictions
with open('/content/vqa_test_predictions.json') as f:
    predictions = json.load(f)

# Load ground truths
with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Map image -> item
gt_dict = {item["image"]: item for item in ground_truths}

# Bộ đếm cho từng loại
categories = {
    "overall": [],
    "yes/no": [],
    "number": [],
    "other": [],
    "unanswerable": []
}

for pred in predictions:
    image_id = pred["image"]
    pred_answer = pred["predicted_answer"].strip().lower()

    if image_id not in gt_dict:
        continue

    gt_item = gt_dict[image_id]
    answer_type = gt_item["answer_type"].lower()
    answerable = gt_item["answerable"]

    # Ground truth answers (10 câu)
    gt_answers = [
        ans["answer"].strip().lower()
        for ans in gt_item["answers"]
        if "answer" in ans
    ]

    # Đếm số câu trùng
    matching = sum(1 for ans in gt_answers if ans == pred_answer)

    # Công thức chính thức của VizWiz
    acc = min(matching / 3, 1.0)

    # Overall
    categories["overall"].append(acc)

    # Unanswerable trường hợp đặc biệt
    if answerable == 0:
        categories["unanswerable"].append(acc)
    else:
        # Phân loại theo answer_type
        if answer_type == "yes/no":
            categories["yes/no"].append(acc)
        elif answer_type == "number":
            categories["number"].append(acc)
        else:
            categories["other"].append(acc)

# Tính trung bình
for cat in categories:
    scores = categories[cat]
    avg = sum(scores)/len(scores) if scores else 0.0
    print(f"{cat.capitalize():15s}: {avg:.4f}")



Overall        : 0.5417
Yes/no         : 0.3846
Number         : 0.3542
Other          : 0.3294
Unanswerable   : 0.9829
